# Notebook 02: Feature Extraction

## Goal
For each prompt in our taxonomy, extract the SAE feature activation vector
from target layers of the loaded model. Save results for analysis in Notebook 03.

## What we're computing
For each (prompt, layer) pair:
1. Run the prompt through the model (GPT-2 Small or Gemma 2 2B, set by Notebook 01)
2. Grab the residual stream at the target layer, last token position
3. Project through the pre-trained SAE to get sparse feature activations

**Result**: A matrix of shape `[n_prompts, n_features]` per layer, saved as `.npz` files.

In [6]:
import torch
import numpy as np
import json
from pathlib import Path
from tqdm import tqdm

# Patch transformers compat before importing transformer_lens
import transformers
for _cls in ("BertForPreTraining", "T5ForConditionalGeneration"):
    if not hasattr(transformers, _cls):
        setattr(transformers, _cls, type(_cls, (), {}))

from transformer_lens import HookedTransformer
from sae_lens import SAE

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

RESULTS_DIR = Path("results/feature_activations")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with open("results/config.json", encoding="utf-8") as f:
    cfg = json.load(f)

MODEL_NAME    = cfg["model_name"]
MODEL_DEVICE  = cfg["model_device"]
MODEL_DTYPE   = cfg["model_dtype"]
TARGET_LAYERS = cfg["target_layers"]
SAE_RELEASE   = cfg["sae_release"]
SAE_WIDTH     = cfg["sae_width"]

HOOK_SUFFIX = "hook_resid_pre" if "gpt2" in SAE_RELEASE else "hook_resid_post"

print(f"Model:         {MODEL_NAME}")
print(f"Device:        {MODEL_DEVICE}")
print(f"Target layers: {TARGET_LAYERS}")
print(f"SAE release:   {SAE_RELEASE}")
print(f"Hook:          blocks.N.{HOOK_SUFFIX}")


Model:         gpt2
Device:        cuda
Target layers: [2, 6, 10]
SAE release:   gpt2-small-res-jb
Hook:          blocks.N.hook_resid_pre


In [7]:
print(f'Loading {MODEL_NAME} on {MODEL_DEVICE}...')

model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    dtype=MODEL_DTYPE,
    fold_ln=False,
    center_writing_weights=False,
    center_unembed=False,
    device=MODEL_DEVICE,
)
model.eval()
print('Model loaded.')

Loading gpt2 on cuda...
Loaded pretrained model gpt2 into HookedTransformer
Model loaded.


In [8]:
# Load one SAE per target layer
# SAE.from_pretrained now returns just the SAE object (no longer a 3-tuple)
saes = {}
for layer in TARGET_LAYERS:
    if 'gpt2' in SAE_RELEASE:
        sae_id = f'blocks.{layer}.hook_resid_pre'
    else:
        sae_id = f'layer_{layer}/width_16k/average_l0_71'

    print(f'Loading SAE layer {layer}  ({sae_id})...')
    sae = SAE.from_pretrained(
        release=SAE_RELEASE,
        sae_id=sae_id,
        device=MODEL_DEVICE,
    )
    sae.eval()
    saes[layer] = sae
    print(f'  {sae.cfg.d_sae} features')

print('\nAll SAEs loaded.')

Loading SAE layer 2  (blocks.2.hook_resid_pre)...
  24576 features
Loading SAE layer 6  (blocks.6.hook_resid_pre)...
  24576 features
Loading SAE layer 10  (blocks.10.hook_resid_pre)...
  24576 features

All SAEs loaded.


In [9]:
# Load prompt taxonomy
with open("data/prompts.json", encoding="utf-8") as f:
    prompt_taxonomy = json.load(f)

categories = list(prompt_taxonomy.keys())
print(f"Categories: {categories}")
for cat in categories:
    print(f"  {cat}: {len(prompt_taxonomy[cat])} prompts")
print(f"Total: {sum(len(v) for v in prompt_taxonomy.values())} prompts")


Categories: ['math', 'code', 'factual', 'creative', 'emotional', 'reasoning']
  math: 50 prompts
  code: 50 prompts
  factual: 50 prompts
  creative: 50 prompts
  emotional: 50 prompts
  reasoning: 50 prompts
Total: 300 prompts


In [10]:
def extract_features(model, sae, prompt, layer):
    """
    Extract SAE feature activations for a single prompt at a given layer.
    Returns np.ndarray of shape [n_features] — sparse, mostly zeros.
    """
    tokens = model.to_tokens(prompt)
    hook_name = f'blocks.{layer}.{HOOK_SUFFIX}'

    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=hook_name)

    # Last token position; cast to float32 — SAE encoder expects it
    resid = cache[hook_name][0, -1, :].float()

    with torch.no_grad():
        feature_acts = sae.encode(resid.unsqueeze(0)).squeeze(0)

    return feature_acts.cpu().numpy()

print('Extraction function defined.')

Extraction function defined.


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Main extraction loop
# Runs all prompts across all layers — this is the expensive step
# Expected time: ~2-5 min on GPU, ~30-60 min on CPU
# ─────────────────────────────────────────────────────────────────────────────

# results[layer][category] = np.ndarray of shape [n_prompts, n_features]
all_results = {layer: {} for layer in TARGET_LAYERS}

for layer in TARGET_LAYERS:
    sae = saes[layer]
    print(f'\n=== Layer {layer} ===')
    
    for category in categories:
        prompts = prompt_taxonomy[category]
        category_features = []
        
        for prompt in tqdm(prompts, desc=f'  {category}'):
            features = extract_features(model, sae, prompt, layer)
            category_features.append(features)
        
        # Stack: [n_prompts, n_features]
        all_results[layer][category] = np.stack(category_features)
        
        # Print sparsity stats
        mean_active = (all_results[layer][category] > 0).sum(axis=1).mean()
        print(f'    {category}: avg {mean_active:.1f} active features')
    
    # Save layer results to disk
    save_path = RESULTS_DIR / f'layer_{layer}.npz'
    np.savez(save_path, **all_results[layer])
    print(f'  Saved to {save_path}')

print('\n✓ Extraction complete.')


=== Layer 2 ===


  math: 100%|████████████████████████████████████████████████████████| 50/50 [00:03<00:00, 13.86it/s]


    math: avg 34.1 active features


  code: 100%|████████████████████████████████████████████████████████| 50/50 [00:02<00:00, 20.82it/s]


    code: avg 344.2 active features


...

    creative: avg 172.2 active features


  emotional: 100%|███████████████████████████████████████████████████| 50/50 [00:02<00:00, 20.82it/s]


    emotional: avg 204.7 active features


  reasoning: 100%|███████████████████████████████████████████████████| 50/50 [00:02<00:00, 20.77it/s]

    reasoning: avg 168.2 active features
  Saved to results\feature_activations\layer_10.npz

✓ Extraction complete.


In [12]:
print("Verifying saved results...")
with open("data/prompts.json", encoding="utf-8") as f:
    categories = list(json.load(f).keys())
for layer in TARGET_LAYERS:
    loaded = np.load(RESULTS_DIR / f"layer_{layer}.npz")
    print(f"Layer {layer}:")
    for cat in categories:
        arr = loaded[cat]
        print(f"  {cat}: shape={arr.shape}, nonzero_mean={np.mean(arr > 0):.4f}")
print("Verification complete. Proceed to Notebook 03.")


Verifying saved results...
Layer 2:
  math: shape=(50, 24576), nonzero_mean=0.0014
  code: shape=(50, 24576), nonzero_mean=0.0140
  factual: shape=(50, 24576), nonzero_mean=0.0011
  creative: shape=(50, 24576), nonzero_mean=0.0053
  emotional: shape=(50, 24576), nonzero_mean=0.0131
  reasoning: shape=(50, 24576), nonzero_mean=0.0033
Layer 6:
  math: shape=(50, 24576), nonzero_mean=0.0025
  code: shape=(50, 24576), nonzero_mean=0.0067
  factual: shape=(50, 24576), nonzero_mean=0.0024
  creative: shape=(50, 24576), nonzero_mean=0.0030
  emotional: shape=(50, 24576), nonzero_mean=0.0060
  reasoning: shape=(50, 24576), nonzero_mean=0.0028
Layer 10:
  math: shape=(50, 24576), nonzero_mean=0.0065
  code: shape=(50, 24576), nonzero_mean=0.0075
  factual: shape=(50, 24576), nonzero_mean=0.0064
  creative: shape=(50, 24576), nonzero_mean=0.0070
  emotional: shape=(50, 24576), nonzero_mean=0.0083
  reasoning: shape=(50, 24576), nonzero_mean=0.0068
Verification complete. Proceed to Notebook 03.
